# LangChain Chains: forma clásica vs forma actual con LCEL


> En LangChain moderno, muchas abstracciones clásicas siguen siendo útiles para entender la historia del framework, pero para código nuevo conviene usar `prompt | modelo | parser`, `RunnableParallel`, `RunnablePassthrough`, `.invoke()`, `.batch()`, `.stream()` y composición explícita.


In [ ]:
# Instalación recomendada para el curso
# Ejecutar una vez en un entorno limpio.

%pip install -U langchain langchain-openai langchain-core langchain-classic python-dotenv pandas


## 1. API Key y configuración


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        "Falta OPENAI_API_KEY. Configurala como variable de entorno o en un archivo .env"
    )

# Cambiá el modelo desde el entorno sin tocar el notebook.
MODEL_NAME = os.environ.get("OPENAI_MODEL", "gpt-5.4-nano-2026-03-17")
print("Modelo:", MODEL_NAME)


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

llm = ChatOpenAI(model=MODEL_NAME, temperature=0.2)
parser = StrOutputParser()


## 2. Ejemplo base: `LLMChain` clásica

### Forma vieja / legacy

En versiones viejas se usaba `LLMChain`. En LangChain v1, las chains legacy viven en `langchain-classic`.



In [ ]:
# Forma clásica: LLMChain
# Requiere: pip install langchain-classic

from langchain_classic.chains import LLMChain

prompt_legacy = ChatPromptTemplate.from_template(
    "Dame 5 nombres creativos para una empresa que fabrica {producto}."
)

legacy_chain = LLMChain(
    llm=llm,
    prompt=prompt_legacy,
    output_parser=parser,
)

legacy_chain.invoke({"producto": "mate eléctrico inteligente"})


### Forma nueva: LCEL

La cadena moderna se escribe componiendo objetos con el operador `|`.

La idea mental es:

`datos de entrada -> prompt -> modelo -> parser -> resultado`


In [ ]:
prompt = ChatPromptTemplate.from_template(
    "Dame 5 nombres creativos para una empresa que fabrica {producto}."
)

chain = prompt | llm | parser

chain.invoke({"producto": "mate eléctrico inteligente"})


## 3. `.invoke()`, `.batch()` y `.stream()`

Con LCEL, la misma chain tiene métodos estándar.


In [ ]:
# Una llamada
chain.invoke({"producto": "bicicletas plegables"})


In [ ]:
# Varias llamadas en lote
productos = [
    {"producto": "zapatillas recicladas"},
    {"producto": "cafeteras portátiles"},
    {"producto": "sensores para huertas urbanas"},
]

respuestas = chain.batch(productos)
for producto, respuesta in zip(productos, respuestas):
    print("\n---", producto["producto"], "---")
    print(respuesta)


In [ ]:
# Streaming de tokens/chunks
for chunk in chain.stream({"producto": "robot educativo para chicos"}):
    print(chunk, end="")


## 4. `SimpleSequentialChain`: clásico vs LCEL

Ejemplo: primero generamos el nombre de la empresa y después un slogan.


In [ ]:
# Forma vieja / legacy
from langchain_classic.chains import SimpleSequentialChain

name_prompt_legacy = ChatPromptTemplate.from_template(
    "Dame un único nombre de empresa para un producto: {producto}. Solo el nombre."
)
slogan_prompt_legacy = ChatPromptTemplate.from_template(
    "Crea un slogan breve para esta empresa: {empresa}."
)

name_chain_legacy = LLMChain(llm=llm, prompt=name_prompt_legacy, output_parser=parser)
slogan_chain_legacy = LLMChain(llm=llm, prompt=slogan_prompt_legacy, output_parser=parser)

simple_seq_legacy = SimpleSequentialChain(
    chains=[name_chain_legacy, slogan_chain_legacy],
    verbose=True,
)

simple_seq_legacy.invoke("auriculares con traducción simultánea")


In [ ]:
# Forma nueva / LCEL

name_prompt = ChatPromptTemplate.from_template(
    "Dame un único nombre de empresa para un producto: {producto}. Solo el nombre."
)
slogan_prompt = ChatPromptTemplate.from_template(
    "Crea un slogan breve para esta empresa: {empresa}."
)

name_chain = name_prompt | llm | parser
slogan_chain = slogan_prompt | llm | parser

# Adaptamos la salida de la primera chain para que sea la entrada esperada por la segunda.
sequential_chain = name_chain | (lambda nombre: {"empresa": nombre}) | slogan_chain

sequential_chain.invoke({"producto": "auriculares con traducción simultánea"})


## 5. `SequentialChain` con múltiples variables

Ejemplo: generamos una reseña, extraemos idioma, hacemos resumen y traducción.


In [ ]:
# Forma vieja / legacy
from langchain_classic.chains import SequentialChain

review_prompt_legacy = ChatPromptTemplate.from_template(
    "Escribí una reseña breve en español sobre este producto: {producto}."
)
language_prompt_legacy = ChatPromptTemplate.from_template(
    "Detectá el idioma del siguiente texto. Respondé solo el idioma:\n\n{reseña}"
)
summary_prompt_legacy = ChatPromptTemplate.from_template(
    "Resumí en una oración la siguiente reseña:\n\n{reseña}"
)
translation_prompt_legacy = ChatPromptTemplate.from_template(
    "Traducí al inglés este resumen escrito en {idioma}:\n\n{resumen}"
)

review_chain_legacy = LLMChain(
    llm=llm, prompt=review_prompt_legacy, output_key="reseña", output_parser=parser
)
language_chain_legacy = LLMChain(
    llm=llm, prompt=language_prompt_legacy, output_key="idioma", output_parser=parser
)
summary_chain_legacy = LLMChain(
    llm=llm, prompt=summary_prompt_legacy, output_key="resumen", output_parser=parser
)
translation_chain_legacy = LLMChain(
    llm=llm, prompt=translation_prompt_legacy, output_key="traduccion", output_parser=parser
)

seq_legacy = SequentialChain(
    chains=[review_chain_legacy, language_chain_legacy, summary_chain_legacy, translation_chain_legacy],
    input_variables=["producto"],
    output_variables=["reseña", "idioma", "resumen", "traduccion"],
    verbose=True,
)

seq_legacy.invoke({"producto": "una mochila antirrobo con cargador solar"})

In [ ]:
# Forma nueva / LCEL con asignaciones explícitas

review_prompt = ChatPromptTemplate.from_template(
    "Escribí una reseña breve en español sobre este producto: {producto}."
)
language_prompt = ChatPromptTemplate.from_template(
    "Detectá el idioma del siguiente texto. Respondé solo el idioma:\n\n{reseña}"
)
summary_prompt = ChatPromptTemplate.from_template(
    "Resumí en una oración la siguiente reseña:\n\n{reseña}"
)
translation_prompt = ChatPromptTemplate.from_template(
    "Traducí al inglés este resumen escrito en {idioma}:\n\n{resumen}"
)

review_chain = review_prompt | llm | parser
language_chain = language_prompt | llm | parser
summary_chain = summary_prompt | llm | parser
translation_chain = translation_prompt | llm | parser

modern_seq = (
    RunnablePassthrough.assign(reseña=review_chain)
    .assign(idioma=language_chain)
    .assign(resumen=summary_chain)
    .assign(traduccion=translation_chain)
)

modern_seq.invoke({"producto": "una mochila antirrobo con cargador solar"})

## 6. Ramas paralelas con `RunnableParallel`

LCEL facilita ejecutar transformaciones independientes sobre la misma entrada.


In [ ]:
sentiment_prompt = ChatPromptTemplate.from_template(
    "Clasificá el sentimiento de esta reseña como positivo, neutral o negativo:\n\n{texto}"
)
keywords_prompt = ChatPromptTemplate.from_template(
    "Extraé 5 palabras clave de esta reseña:\n\n{texto}"
)
summary_prompt = ChatPromptTemplate.from_template(
    "Resumí esta reseña en una oración:\n\n{texto}"
)

analysis_parallel = RunnableParallel(
    sentimiento=sentiment_prompt | llm | parser,
    keywords=keywords_prompt | llm | parser,
    resumen=summary_prompt | llm | parser,
)

texto = "La batería dura muchísimo y el diseño es excelente, aunque la app móvil todavía tiene errores."
analysis_parallel.invoke({"texto": texto})

## 7. Transformaciones intermedias con `RunnableLambda`

Útil para limpiar entradas, normalizar formatos o crear campos derivados.


In [ ]:
def limpiar_producto(input_dict: dict) -> dict:
    producto = input_dict["producto"].strip().lower()
    return {"producto": producto}

cleaning_chain = (
    RunnableLambda(limpiar_producto)
    | ChatPromptTemplate.from_template("Proponé un nombre premium para: {producto}")
    | llm
    | parser
)

cleaning_chain.invoke({"producto": "   BOTELLA TÉRMICA CON SENSOR DE TEMPERATURA   "})


## 8. Salida estructurada con Pydantic

Para producción, muchas veces conviene pedir objetos estructurados en lugar de texto libre.


In [ ]:
from pydantic import BaseModel, Field

class IdeaProducto(BaseModel):
    nombre: str = Field(description="Nombre comercial sugerido")
    slogan: str = Field(description="Slogan breve")
    publico_objetivo: str = Field(description="Segmento de clientes ideal")

structured_llm = llm.with_structured_output(IdeaProducto)

structured_prompt = ChatPromptTemplate.from_template(
    "Generá una idea de branding para un producto: {producto}"
)

structured_chain = structured_prompt | structured_llm

idea = structured_chain.invoke({"producto": "cuaderno reutilizable con IA"})
idea
